# DeepLearning.AI

## Tagging and Extraction Using OpenAI fñunctions

<img src="./pics/functions-tools-agents-langchain.png" width=800> 

https://learn.deeplearning.ai/courses/functions-tools-agents-langchain/information

In [1]:
pwd

'c:\\Users\\crodr\\aiDeepLearning\\functions-tools-agents-langchain'

In [2]:
!python --version


Python 3.13.13


In [3]:
import os
import openai
from dotenv import load_dotenv

load_dotenv()
openai.api_key = os.environ["OPENAI_API_KEY"]

In [4]:
from typing import List
from pydantic import BaseModel, Field
from langchain_core.utils.function_calling import convert_to_openai_function

In [5]:
class Tagging(BaseModel):
    """Tag the piece of text with particular info."""

    sentiment: str = Field(
        description="sentiment of text, should be `pos`, `neg`, or `neutral`"
    )
    language: str = Field(description="language of text (should be ISO 639-1 code)")

In [6]:
convert_to_openai_function(Tagging)

{'name': 'Tagging',
 'description': 'Tag the piece of text with particular info.',
 'parameters': {'properties': {'sentiment': {'description': 'sentiment of text, should be `pos`, `neg`, or `neutral`',
    'type': 'string'},
   'language': {'description': 'language of text (should be ISO 639-1 code)',
    'type': 'string'}},
  'required': ['sentiment', 'language'],
  'type': 'object'}}

In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI


In [8]:
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)


In [9]:
tagging_functions = [convert_to_openai_function(Tagging)]

In [10]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Think carefully, and then tag the text as instructed"),
        ("user", "{input}"),
    ]
)

In [11]:
model_with_functions = model.bind(
    functions=tagging_functions, function_call={"name": "Tagging"}
)

In [12]:
tagging_chain = prompt | model_with_functions

In [13]:
tagging_chain.invoke({"input": "I love langchain"})

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"sentiment":"pos","language":"en"}', 'name': 'Tagging'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 105, 'total_tokens': 115, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_38fd64cff5', 'id': 'chatcmpl-De1SR4wBpotppRbl4As3mKepmtCMZ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e12c0-ecf1-79f2-b382-563d35d9e939-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 105, 'output_tokens': 10, 'total_tokens': 115, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [14]:
tagging_chain.invoke({"input": "non mi piace questo cibo"})


AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"sentiment":"neg","language":"it"}', 'name': 'Tagging'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 107, 'total_tokens': 117, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e2d886d409', 'id': 'chatcmpl-De1SUHEdkcljMNSvWShpMkv7GLbNX', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e12c0-fc31-7811-85ef-f11db3dc5d68-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 107, 'output_tokens': 10, 'total_tokens': 117, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [15]:
from langchain_core.output_parsers.openai_functions import JsonOutputFunctionsParser

In [16]:
tagging_chain = prompt | model_with_functions | JsonOutputFunctionsParser()

In [17]:
tagging_chain.invoke({"input": "non mi piace questo cibo"})

{'sentiment': 'neg', 'language': 'it'}

In [18]:
tagging_chain.invoke({"input": "no me gusta esta comida"})


{'sentiment': 'neg', 'language': 'es'}

### Extraction

Extraction is similar to tagging, but used for extracting multiple pieces of information

In [19]:
from typing import Optional


class Person(BaseModel):
    """Information about a person."""

    name: str = Field(description="person's name")
    age: Optional[int] = Field(description="person's age")

In [20]:
class Information(BaseModel):
    """Information to extract."""

    people: List[Person] = Field(description="List of info about people")


In [21]:
convert_to_openai_function(Information)

{'name': 'Information',
 'description': 'Information to extract.',
 'parameters': {'properties': {'people': {'description': 'List of info about people',
    'items': {'description': 'Information about a person.',
     'properties': {'name': {'description': "person's name", 'type': 'string'},
      'age': {'anyOf': [{'type': 'integer'}, {'type': 'null'}],
       'description': "person's age"}},
     'required': ['name', 'age'],
     'type': 'object'},
    'type': 'array'}},
  'required': ['people'],
  'type': 'object'}}

In [22]:
extraction_functions = [convert_to_openai_function(Information)]
extraction_model = model.bind(
    functions=extraction_functions, function_call={"name": "Information"}
)

In [23]:
extraction_model.invoke("Joe is 30, his mom is Martha")

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"people":[{"name":"Joe","age":30},{"name":"Martha","age":null}]}', 'name': 'Information'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 89, 'total_tokens': 111, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_d88f6e55bb', 'id': 'chatcmpl-De1Szt5gybXNpbG0YSdjDcUSr7SRX', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e12c1-75c6-7da3-85a4-1955a36f92ed-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 89, 'output_tokens': 22, 'total_tokens': 111, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [24]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Extract the relevant information, if not explicitly provide do not guess. Extract partial info",
        ),
        ("human", "{input}"),
    ]
)

In [25]:
extraction_chain = prompt | extraction_model

In [26]:
extraction_chain.invoke("Joe is 30, his mom is Martha")


AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"people":[{"name":"Joe","age":30},{"name":"Martha","age":null}]}', 'name': 'Information'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 106, 'total_tokens': 128, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_aa5a83636e', 'id': 'chatcmpl-De1T9RyyN7HVm3TaPmqEP9qim5i57', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e12c1-9c56-77c2-96bd-5107e7a79bae-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 106, 'output_tokens': 22, 'total_tokens': 128, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [27]:
extraction_chain = prompt | extraction_model | JsonOutputFunctionsParser()

In [28]:
extraction_chain.invoke("Joe is 30, his mom is Martha")


{'people': [{'name': 'Joe', 'age': 30}, {'name': 'Martha', 'age': None}]}

In [29]:
extraction_chain = (
    prompt | extraction_model | JsonOutputFunctionsParser(key_name="people")
)

In [30]:
extraction_chain.invoke({"input": "Joe is 30, his mom is Martha"})


{'people': [{'name': 'Joe', 'age': 30}, {'name': 'Martha', 'age': None}]}

### Doing for real

We can apply tagging to a larger body of text.

For example, let's load this blog post and extract tag information from a sub-set of the text

In [ ]:
import sys

!uv pip install beautifulsoup4 langchain-community langchain-openai langchain-core

Checked 4 packages in 23ms


In [ ]:
# import bs4
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
documents = loader.load()


USER_AGENT environment variable not set, consider setting it to identify your requests.


In [33]:
doc = documents[0]

In [34]:
page_content = doc.page_content[:10_000]

In [35]:
print(page_content[:10_000])







LLM Powered Autonomous Agents | Lil'Log







































Lil'Log

















|






Posts




Archive




Search




Tags




FAQ









      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


 


Table of Contents



Agent System Overview

Component One: Planning

Task Decomposition

Self-Reflection


Component Two: Memory

Types of Memory

Maximum Inner Product Search (MIPS)


Component Three: Tool Use

Case Studies

Scientific Discovery Agent

Generative Agents Simulation

Proof-of-Concept Examples


Challenges

Citation

References





Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.


In [ ]:
class Overview(BaseModel):
    """Overview of a section text."""

    summary: str = Field(description="Provide a concise summary of the content.")
    language: str = Field(
        description="Provide the language of the content is written in."
    )
    keywords: str = Field(description="Provide Kkeywords related to the content.")


In [ ]:
overview_tagging_function = [convert_to_openai_function(Overview)]
tagging_model = model.bind(
    functions=overview_tagging_function, function_call={"name": "Overview"}
)
tagging_chain = prompt | tagging_model | JsonOutputFunctionsParser()

In [38]:
tagging_chain.invoke({"input": page_content})

{'summary': 'The article discusses the concept of LLM (large language model) powered autonomous agents, detailing their core components such as planning, memory, and tool use. It highlights the importance of task decomposition, self-reflection, and the integration of memory types for enhancing agent performance. Various methodologies like Chain of Thought (CoT), Tree of Thoughts, and ReAct are explored for improving task handling and decision-making. The article also addresses challenges and provides examples of proof-of-concept agents like AutoGPT and BabyAGI.',
 'language': 'English',
 'keywords': 'LLM, autonomous agents, planning, memory, task decomposition, self-reflection, tool use, Chain of Thought, ReAct, proof-of-concept'}

In [ ]:
class Paper(BaseModel):
    """Information about papers mentioned."""

    title: str
    author: Optional[str]


class Info(BaseModel):
    """Information to extract"""

    papers: List[Paper]

In [ ]:
from langchain_core.output_parsers.openai_functions import JsonKeyOutputFunctionsParser

In [ ]:
paper_extraction_function = [convert_to_openai_function(Info)]
extraction_model = model.bind(
    functions=paper_extraction_function, function_call={"name": "Info"}
)
extraction_chain = (
    prompt | extraction_model | JsonKeyOutputFunctionsParser(key_name="papers")
)

In [45]:
extraction_chain.invoke({"input": page_content})

[{'title': 'Chain of Thought', 'author': 'Wei et al. 2022'},
 {'title': 'Tree of Thoughts', 'author': 'Yao et al. 2023'},
 {'title': 'LLM+P', 'author': 'Liu et al. 2023'},
 {'title': 'ReAct', 'author': 'Yao et al. 2023'},
 {'title': 'Reflexion', 'author': 'Shinn & Labash 2023'},
 {'title': 'Chain of Hindsight', 'author': 'Liu et al. 2023'},
 {'title': 'Algorithm Distillation', 'author': 'Laskin et al. 2023'},
 {'title': 'RL^2', 'author': 'Duan et al. 2017'}]

In [ ]:
template = """A article will be passed to you. Extract from it all papers that are mentioned by this article folow by its author.

Do not extract the name of the article itself. If no papers are mentioned that's fine - you don't need to extract any! Just return an empty list.

Do not make up or guess ANY extra information. Only extract what exactly is in the text."""

prompt = ChatPromptTemplate.from_messages([("system", template), ("human", "{input}")])

In [ ]:
extraction_chain = (
    prompt | extraction_model | JsonKeyOutputFunctionsParser(key_name="papers")
)


In [48]:
extraction_chain.invoke({"input": page_content})

[{'title': 'Chain of thought', 'author': 'Wei et al. 2022'},
 {'title': 'Tree of Thoughts', 'author': 'Yao et al. 2023'},
 {'title': 'LLM+P', 'author': 'Liu et al. 2023'},
 {'title': 'ReAct', 'author': 'Yao et al. 2023'},
 {'title': 'Reflexion', 'author': 'Shinn & Labash 2023'},
 {'title': 'Chain of Hindsight', 'author': 'Liu et al. 2023'},
 {'title': 'Algorithm Distillation', 'author': 'Laskin et al. 2023'},
 {'title': 'RL^2', 'author': 'Duan et al. 2017'}]

In [49]:
extraction_chain.invoke({"input": "hi"})


[]

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_overlap=0)

In [53]:
splits = text_splitter.split_text(doc.page_content)

In [54]:
len(splits)

15

In [55]:
def flatten(matrix):
    flat_list = []
    for row in matrix:
        flat_list += row
    return flat_list

In [57]:
flatten([[1, 2], [3, 4]])

[1, 2, 3, 4]

In [58]:
print(splits[0])

LLM Powered Autonomous Agents | Lil'Log







































Lil'Log

















|






Posts




Archive




Search




Tags




FAQ









      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


 


Table of Contents



Agent System Overview

Component One: Planning

Task Decomposition

Self-Reflection


Component Two: Memory

Types of Memory

Maximum Inner Product Search (MIPS)


Component Three: Tool Use

Case Studies

Scientific Discovery Agent

Generative Agents Simulation

Proof-of-Concept Examples


Challenges

Citation

References





Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
Agent 

In [60]:
from langchain_core.runnables import RunnableLambda

In [ ]:
prep = RunnableLambda(lambda x: [{"input": doc} for doc in text_splitter.split_text(x)])

In [62]:
prep.invoke("hi")

[{'input': 'hi'}]

In [63]:
chain = prep | extraction_chain.map() | flatten

In [64]:
chain.invoke(doc.page_content)

[{'title': 'Chain of thought', 'author': 'Wei et al. 2022'},
 {'title': 'Tree of Thoughts', 'author': 'Yao et al. 2023'},
 {'title': 'LLM+P', 'author': 'Liu et al. 2023'},
 {'title': 'ReAct', 'author': 'Yao et al. 2023'},
 {'title': 'Reflexion', 'author': 'Shinn & Labash 2023'},
 {'title': 'Chain of Hindsight', 'author': 'Liu et al. 2023'},
 {'title': 'Algorithm Distillation', 'author': 'Laskin et al. 2023'},
 {'title': 'Shinn & Labash', 'author': '2023'},
 {'title': 'RL^2', 'author': 'Duan et al. 2017'},
 {'title': 'A3C', 'author': None},
 {'title': 'DQN', 'author': None},
 {'title': 'Laskin et al. 2023', 'author': None},
 {'title': 'Miller', 'author': '1956'},
 {'title': 'MRKL', 'author': 'Karpas et al. 2022'},
 {'title': 'TALM', 'author': 'Parisi et al. 2022'},
 {'title': 'Toolformer', 'author': 'Schick et al. 2023'},
 {'title': 'HuggingGPT', 'author': 'Shen et al. 2023'},
 {'title': 'API-Bank', 'author': 'Li et al. 2023'},
 {'title': 'ChemCrow', 'author': 'Bran et al. 2023'},
 {'ti